# 📓 Exemplo 11: O ritmo da mudança (Mann-Kendall e Sen)

**Missão 10** · [![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcoliveira-utfpr/change_climate/blob/main/code/notebooks_exemplo/11_tendencia_ritmo.ipynb)

> **Pergunta:** no ritmo atual, em quantos anos cada critério cruza o limiar?

### 🧭 Como usar este notebook
1. 💾 **Faça uma cópia:** menu **Arquivo → Salvar uma cópia no Drive**. Mova a cópia para a pasta
   `mudanca_climatica_br/notebooks/`. Assim você pode mudar tudo sem estragar o original.
2. ▶️ **Rode uma célula por vez** (botão ▶ ou **Shift + Enter**) e **leia os comentários** (as linhas com `#`).
3. 🧪 Os exemplos usam **Santa Helena-PR** e **Curitiba-PR**.
4. ✍️ No fim, a seção **Sua vez** pede para você **replicar o método para as capitais do Brasil**.

In [ ]:
# Instala as bibliotecas que não vêm no Colab (só precisa rodar uma vez por sessão)
!pip install agrometeorologiapy pymannkendall

In [ ]:
# Endereço da pasta de dados do projeto no GitHub (arquivos "raw" = texto puro).
# O pandas consegue ler um CSV direto de um endereço da internet.
URL_DADOS = 'https://raw.githubusercontent.com/fcoliveira-utfpr/change_climate/main/code/dados'

In [ ]:
import pandas as pd

# Lê as séries mensais (1961-2025) direto do GitHub.
#   parse_dates=['data'] -> a coluna 'data' é entendida como data
#   index_col='data'     -> a coluna 'data' vira o índice (o "nome" de cada linha)
serie_sh = pd.read_csv(f'{URL_DADOS}/santa_helena_mensal.csv', parse_dates=['data'], index_col='data')
serie_ct = pd.read_csv(f'{URL_DADOS}/curitiba_mensal.csv', parse_dates=['data'], index_col='data')

LATITUDE_SH = -24.86
LATITUDE_CT = -25.43

serie_sh.head()

**Anos até cruzar = folga ÷ ritmo.** O ritmo é medido na **série anual** (um valor por ano), nunca na
janela móvel.
- **Mann-Kendall:** existe tendência? (p < 0,05 = significativa)
- **Declividade de Sen:** quanto muda por ano?

## 1. Séries anuais dos critérios

In [ ]:
# Temperatura do mês mais frio e do mês mais quente de CADA ano
t_mes_frio_anual_sh = serie_sh['tmed'].resample('YS').min()
t_mes_quente_anual_ct = serie_ct['tmed'].resample('YS').max()

t_mes_frio_anual_sh.head()

## 2. Mann-Kendall e Sen

In [ ]:
import pymannkendall as mk

teste_sh = mk.original_test(t_mes_frio_anual_sh)
print('Santa Helena, mês mais frio')
print('   tendência:', teste_sh.trend, '| p-valor:', round(teste_sh.p, 4))
print('   ritmo:', round(teste_sh.slope * 10, 3), '°C por década')

# Teste corrigido para a "memória" da série (autocorrelação)
teste_sh_corrigido = mk.hamed_rao_modification_test(t_mes_frio_anual_sh)
print('   p-valor corrigido (Hamed-Rao):', round(teste_sh_corrigido.p, 4))

In [ ]:
teste_ct = mk.original_test(t_mes_quente_anual_ct)
print('Curitiba, mês mais quente')
print('   tendência:', teste_ct.trend, '| p-valor:', round(teste_ct.p, 4))
print('   ritmo:', round(teste_ct.slope * 10, 3), '°C por década')

## 3. Quantos anos faltam? (função com as 4 situações)

In [ ]:
def anos_ate_cruzar(folga_atual, ritmo_por_ano):
    """
    Devolve os anos até cruzar o limiar, no ritmo atual, ou um texto explicando por que não cruza.
    folga_atual: folga hoje (negativa = abaixo do limiar)
    ritmo_por_ano: declividade de Sen, em unidades por ano
    """
    if folga_atual < 0 and ritmo_por_ano > 0:
        return round(-folga_atual / ritmo_por_ano)        # abaixo e subindo: vai cruzar
    elif folga_atual > 0 and ritmo_por_ano < 0:
        return round(folga_atual / -ritmo_por_ano)        # acima e descendo: vai cruzar
    elif folga_atual > 0 and ritmo_por_ano > 0:
        return 'acima e se afastando (já cruzou?)'
    else:
        return 'abaixo e se afastando'


# Folga atual = a do clima de 1996-2025 (calculada aqui mesmo, sem depender de outro notebook)
ultima_janela_sh = serie_sh.loc['1996':'2025']
normal_atual_sh = ultima_janela_sh.groupby(ultima_janela_sh.index.month).mean()
folga_A_atual_sh = normal_atual_sh['tmed'].min() - 18

print('Santa Helena, folga A/C atual:', round(folga_A_atual_sh, 2), '°C')
print('Anos até cruzar:', anos_ate_cruzar(folga_A_atual_sh, teste_sh.slope))

> ⚠️ É uma **extrapolação**: supõe que o ritmo continue igual. Escreva "no ritmo observado, cruzaria em
> cerca de X anos", nunca "vai mudar em X anos". E só use o ritmo se a tendência for significativa.

## ✍️ Sua vez: as 27 capitais
1. Para cada capital, escolha o **critério de Köppen mais próximo do zero** em 2025 (do resumo do Exemplo 10).
2. Monte a série anual desse critério, aplique Mann-Kendall e Sen, e calcule os anos até cruzar.
3. Salve a tabela `resultados/ritmo_27_capitais.csv`: capital | critério | folga 2025 | ritmo/década |
   p-valor | anos até cruzar.
4. Quais capitais cruzariam um limiar em menos de 30 anos, no ritmo observado?

In [ ]:
# As capitais que VOCÊ extraiu na Missão 5 (exemplo 06) estão no seu Drive.
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/mudanca_climatica_br'

# Tabela das 27 capitais (uf, capital, regiao, lat, lon), montada por você
capitais = pd.read_csv(f'{BASE}/dados/capitais_27.csv')

# Exemplo: lendo a série de uma capital (sigla em minúsculas)
uf = 'df'
serie_capital = pd.read_csv(f'{BASE}/dados/capitais/{uf}.csv', parse_dates=['data'], index_col='data')
serie_capital.head()